Code created by: Lorena Espinosa, Johana Rátiva, Eduards Chipatecua 

Class: Procesamiento del Lenguaje Natural

University: Universidad de los Andes

Date: August 31, 2026

In [48]:
import numpy as np
import pandas as pd
from pathlib import Path
import re
import unicodedata


from IPython.display import display
from huggingface_hub import snapshot_download

import nltk
from nltk.tokenize import sent_tokenize,word_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

/home/lorena/python/NLP-ml/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/lorena/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package punkt to /home/lorena/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/lorena/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Preparación del corpus

## 1.1 Lectura del Dataset

In [49]:
data_dir = snapshot_download(
    repo_id="jhonrayo99/nlp-tarea-2-ngramas",
    repo_type="dataset",
)

archivos = list(Path(data_dir).glob("*.txt"))
austen_train_files = list(Path(data_dir).glob("austen_train_*.txt"))
twain_train_files = list( Path(data_dir).glob("twain_train_*.txt"))
austen_test_files = list( Path(data_dir).glob("austen_test_*.txt"))
twain_test_files = list( Path(data_dir).glob("twain_test_*.txt"))

Fetching 60 files: 100%|██████████| 60/60 [00:00<00:00, 659.60it/s]


se arma cada database como una lista de string, donde cada string contiene todo el texto interno de cada libro

In [50]:
def cargar_textos(archivos):
    textos = []

    for archivo in archivos:
        with open(archivo, "r", encoding="utf-8") as f:
             textos.append(f.read())

    return textos


austen_train = cargar_textos(austen_train_files)
twain_train = cargar_textos(twain_train_files)

austen_test = cargar_textos(austen_test_files)
twain_test = cargar_textos(twain_test_files)

In [ ]:
resumen = pd.DataFrame({
    "Dataset": ["Train", "Train", "Test", "Test"],
    "Autor": ["Jane Austen", "Mark Twain", "Jane Austen", "Mark Twain"],
    "Obras": [
        len(austen_train_files),
        len(twain_train_files),
        len(austen_test_files),
        len(twain_test_files)
    ],
    "Registros": [
        len(austen_train),
        len(twain_train),
        len(austen_test),
        len(twain_test)
    ]
})

display(resumen)

,Dataset,Autor,Obras,Registros
0,Train,Jane Austen,9,9
1,Train,Mark Twain,44,44
2,Test,Jane Austen,1,1
3,Test,Mark Twain,1,1


## 1.2 Segmentacion, tokenizacion y normalizacion

Los registros del dataset son fragmentos de libros, no necesariamente oraciones completas. Primero unimos los fragmentos de cada autor conservando saltos de linea y luego usamos `sent_tokenize` para detectar los limites de las oraciones. Despues usamos `word_tokenize` para separar cada oracion en palabras y signos de puntuacion.

Normalizamos convirtiendo el texto a minusculas y reemplazando los numeros por `<NUM>`. No eliminamos palabras de parada, no aplicamos stemming y no aplicamos lematizacion. El entrenamiento y la prueba se mantienen separados desde ahora.

no es bueno quitar los espacios en blanco por que quito los finales de linea para cada libro, sent_tockenize divide por libro y los saltos en linea los convierte cada uno en un array indenpendiente, entonces quedo con division por libro y sub division por linea

In [52]:
def tokenizar_libro(libro):
    # 1. Segmentar en oraciones.
    oraciones = sent_tokenize(libro)
    # 2. Separar palabras y signos de puntuacion.
    return [word_tokenize(oracion) for oracion in oraciones]

In [53]:
train_jane_tokens = [tokenizar_libro(libro) for libro in austen_train]
train_twain_tokens = [tokenizar_libro(libro) for libro in twain_train]

test_jane_tokens = [tokenizar_libro(libro) for libro in austen_test]
test_twain_tokens = [tokenizar_libro(libro) for libro in twain_test]

In [54]:
tabla_resumen_tokenizacion = pd.DataFrame({
    "Dataset": [
        "Train Jane Austen",
        "Train Mark Twain",
        "Test Jane Austen",
        "Test Mark Twain"
    ],
    "Libros": [
        len(train_jane_tokens),
        len(train_twain_tokens),
        len(test_jane_tokens),
        len(test_twain_tokens)
    ],
    "Oraciones": [
        sum(len(libro) for libro in train_jane_tokens),
        sum(len(libro) for libro in train_twain_tokens),
        sum(len(libro) for libro in test_jane_tokens),
        sum(len(libro) for libro in test_twain_tokens)
    ],
    "Tokens": [
        sum(len(oracion) for libro in train_jane_tokens for oracion in libro),
        sum(len(oracion) for libro in train_twain_tokens for oracion in libro),
        sum(len(oracion) for libro in test_jane_tokens for oracion in libro),
        sum(len(oracion) for libro in test_twain_tokens for oracion in libro)
    ]
})

display(tabla_resumen_tokenizacion)

,Dataset,Libros,Oraciones,Tokens
0,Train Jane Austen,9,32478,857807
1,Train Mark Twain,44,140517,3305102
2,Test Jane Austen,1,7493,192857
3,Test Mark Twain,1,5961,136716


fullmatch(r"\d+(?:[.,]\d+)*", token) -> esto no funciona por que hace obligatorio numero con para decimal y no identifica numeros de la forma 7½ 

In [55]:
def is_number(token):
    return any(unicodedata.numeric(c, None) is not None for c in token)

def normalizar_libro(oraciones):
    resultado = []

    for oracion in oraciones:
        
        nueva_oracion = []

        for token in oracion:
            if is_number(token):
                #1. Reemplazar cada numero por un unico token especial.
                nueva_oracion.append("<NUM>")
            else:
                #2. Normalizar: convertir a minusculas.
                nueva_oracion.append(token.lower())

        resultado.append(nueva_oracion)

    return resultado

In [56]:
train_jane_tokens = [normalizar_libro(tokenizar_libro(libro))for libro in austen_train]

train_twain_tokens = [normalizar_libro(tokenizar_libro(libro)) for libro in twain_train]

test_jane_tokens = [normalizar_libro(tokenizar_libro(libro))for libro in austen_test]

test_twain_tokens = [ normalizar_libro(tokenizar_libro(libro))for libro in twain_test]

In [57]:
def metricas_normalizacion(normalizado):
    tokens_normalizados = [token for libro in normalizado for oracion in libro for token in oracion]

    return {
        "Tokens": len(tokens_normalizados),
        "<NUM>": tokens_normalizados.count("<NUM>"),
    }
tabla_resumen_normalizacion = pd.DataFrame([
    {"Dataset": "Train Jane Austen", **metricas_normalizacion(train_jane_tokens)},
    {"Dataset": "Train Mark Twain", **metricas_normalizacion(train_twain_tokens)},
    {"Dataset": "Test Jane Austen", **metricas_normalizacion(test_jane_tokens)},
    {"Dataset": "Test Mark Twain", **metricas_normalizacion(test_twain_tokens)}
])

display(tabla_resumen_normalizacion)

,Dataset,Tokens,<NUM>
0,Train Jane Austen,857807,1637
1,Train Mark Twain,3305102,9113
2,Test Jane Austen,192857,9
3,Test Mark Twain,136716,127


In [58]:
austen_train_oraciones = train_jane_tokens
austen_test_oraciones = test_jane_tokens

twain_train_oraciones = train_twain_tokens
twain_test_oraciones = test_twain_tokens

## 1.3 Construccion del vocabulario y reemplazo por `<UNK>`

Un vocabulario es el conjunto de tokens que el modelo reconoce. Se construye usando solamente las oraciones de entrenamiento, porque el conjunto de prueba debe representar texto no visto por el modelo.

En este notebook conservamos los tokens cuya frecuencia es mayor que uno. Esto incluye palabras y signos de puntuacion, porque ambos forman parte de las secuencias que usara el modelo. `<UNK>` se agrega explicitamente al vocabulario para representar cualquier token que no sea conocido. Austen y Twain tienen vocabularios separados.

In [59]:
def construir_vocabulario(oraciones):
    """
    Esta función genera un vocabulario a partir de las palabras que tienen mas de una aparición en las oraciones. 

    Parameters
    -----------
        oraciones: lista de listas de tokens.
    
    Returns
    ------------
        tokens_unicos: Arreglo de tokens que incluye los tokens con frecuencia = 1.
        frecuencias: Arreglo de conteos por token.
        vocabulario: set de strings con el vocabulario depurado.
    """

    # Unir todos los tokens de las oraciones en una sola lista.
    tokens = []

    for oracion in oraciones:
        for token in oracion:
            tokens.append(token)

    # Obtener tokens únicos y sus frecuencias.
    tokens_unicos, frecuencias = np.unique(
        tokens,
        return_counts=True
    )

    # Conservar solamente los tokens con frecuencia mayor que 1.
    vocabulario = set()

    for token, frecuencia in zip(tokens_unicos, frecuencias):
        if frecuencia > 1:
            vocabulario.add(token)

    # <UNK> representa los tokens que el modelo no conoce.
    vocabulario.add("<UNK>")

    return tokens_unicos, frecuencias, vocabulario

In [60]:
austen_train_oraciones = [
    oracion
    for libro in train_jane_tokens
    for oracion in libro
]

twain_train_oraciones = [
    oracion
    for libro in train_twain_tokens
    for oracion in libro
]
austen_test_oraciones = [
    oracion
    for libro in test_jane_tokens
    for oracion in libro
]

twain_test_oraciones = [
    oracion
    for libro in test_twain_tokens
    for oracion in libro
]

In [61]:
## Construcción de los vocabularios 
tokens_austen, frecuencias_austen, vocabulario_austen = construir_vocabulario(austen_train_oraciones)

tokens_twain, frecuencias_twain, vocabulario_twain = construir_vocabulario(twain_train_oraciones)

In [62]:
print("tokens_austen")
print(f"Tipo: {type(tokens_austen)}")
print(f"Primeros elementos: {tokens_austen[:10]}")

print("\nfrecuencias_austen")
print(f"Tipo: {type(frecuencias_austen)}")
print(f"Contenido: {frecuencias_austen}")

print("\nvocabulario_austen")
print(f"Tipo: {type(vocabulario_austen)}")
print(f"Primeros elementos: {list(vocabulario_austen)[:10]}")

tokens_austen
Tipo: <class 'numpy.ndarray'>
Primeros elementos: ['!' '&' "'" "''" "'s" '(' ')' '*' ',' '--']

frecuencias_austen
Tipo: <class 'numpy.ndarray'>
Contenido: [2716  800   33 ... 4287 7333 7263]

vocabulario_austen
Tipo: <class 'set'>
Primeros elementos: [np.str_('comprised'), np.str_('clearly'), np.str_('forbidding'), np.str_('innocent'), np.str_('impatiently'), np.str_('intelligible'), np.str_('retentive'), np.str_('unagreeable'), np.str_('atoned'), np.str_('heavens')]


In [63]:
# ajustar el train para que las palabras con frecuencia 1 sean remplazadas por <UNK>

def reemplazar_unk(oraciones, vocabulario):
    """
    Esta función remplaza los tokens no vistos por <UNK>, para ello usa el vocabulario construido. 

    Parameters
    -----------
        oraciones: lista de listas de tokens.
        vocabulario: set de palabras 
    Returns
    ------------
        oracions_unk: lista de listas de tokens sin palabras desconocidas o con frecuencia menor a 1. Cuenta con tokens de tipo <UNK>
    """
    oraciones_unk = []

    for oracion in oraciones:
        nueva_oracion = []

        for token in oracion:
            if token in vocabulario:
                nueva_oracion.append(token)
            else:
                nueva_oracion.append("<UNK>")

        oraciones_unk.append(nueva_oracion)

    return oraciones_unk

In [64]:
austen_train_unk = reemplazar_unk(austen_train_oraciones,vocabulario_austen)

twain_train_unk = reemplazar_unk(twain_train_oraciones,vocabulario_twain)

austen_test_unk = reemplazar_unk(austen_test_oraciones,vocabulario_austen)

twain_test_unk = reemplazar_unk(twain_test_oraciones,vocabulario_twain)

In [65]:
for i, elem in enumerate(austen_train_unk[0:10]):
    print(f"Índice {i}: {elem}")

Índice 0: ['transcriber', "'s", 'note', '<UNK>', '<UNK>', 'is', 'indicated', 'by', '<UNK>', '.']
Índice 1: ['<UNK>', 'letters', 'are', 'indicated', 'thus', ':', 'm^r', ',', 'm^', '{', 'rs', '}', '.']
Índice 2: ['fragment', 'of', 'a', 'novel', 'by', 'jane', 'austen', '<UNK>', 'impression', '<NUM>', 'fragment', 'of', 'a', 'novel', 'written', 'by', 'jane', 'austen', '<UNK>', '<NUM>', 'now', 'first', 'printed', 'from', 'the', 'manuscript', 'oxford', 'at', 'the', '<UNK>', 'press', '<NUM>', 'oxford', 'university', 'press', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'publisher', 'to', 'the', 'university', 'printed', 'in', 'england', 'preface', 'the', 'fragment', 'of', 'a', 'novel', ',', 'written', 'by', 'jane', 'austen', 'in', 'the', 'first', 'three', 'months', 'of', 'the', 'year', 'in', 'which', 'she', 'died', ',', 'has', 'no', 'name', ';', 'but', 'it', 'has', 'long', 'been', 'known', 'to', '

## 1.4 Agregar `<s>` y `</s>` y preparar frecuencias de unigramas

Agregaremos `<s>` al inicio y `</s>` al final de cada oracion. Las marcas se agregan tanto al entrenamiento como al test.

Despues de reemplazar los tokens de frecuencia uno por `<UNK>` en entrenamiento, volvemos a contar los tokens. Esta es la frecuencia que usara el modelo, porque ahora todas las apariciones de tokens desconocidos estan agrupadas bajo `<UNK>`. El conteo sigue realizandose con `np.unique`; adicionalmente creamos un diccionario para consultar facilmente la frecuencia de un token.

In [66]:
def agregar_marcas(oraciones):
    """Agrega una marca de inicio y una marca de final a cada oracion.

    Parameters
    ----------
    oraciones: list[list[str]]
        Oraciones tokenizadas.

    Returns
    -------
    list[list[str]]
        Oraciones con `<s>` al inicio y `</s>` al final.
    """
    oraciones_marcadas = []

    for oracion in oraciones:
        nueva_oracion = ["<s>"]

        for token in oracion:
            nueva_oracion.append(token)

        nueva_oracion.append("</s>")
        oraciones_marcadas.append(nueva_oracion)

    return oraciones_marcadas


In [67]:
# Agregación de marcas de inicio y fin a las oraciones
austen_train_marcado = agregar_marcas(austen_train_unk)
twain_train_marcado = agregar_marcas(twain_train_unk)

austen_test_marcado = agregar_marcas(austen_test_unk)
twain_test_marcado = agregar_marcas(twain_test_unk)


In [68]:
# Agregación de marcas de inicio y de fin al vocabulario.

# Las marcas forman parte de los tokens que el modelo puede reconocer.
vocabulario_austen.add("<s>")
vocabulario_austen.add("</s>")
vocabulario_twain.add("<s>")
vocabulario_twain.add("</s>")


In [69]:
# Recontar Austen sobre la representacion final del entrenamiento.
tokens_austen_modelo = []
for oracion in austen_train_marcado:
    for token in oracion:
        tokens_austen_modelo.append(token)

tokens_austen_modelo, frecuencias_austen_modelo = np.unique(
    tokens_austen_modelo,
    return_counts=True
)

frecuencias_austen_modelo_dict = {}
for token, frecuencia in zip(tokens_austen_modelo, frecuencias_austen_modelo):
    frecuencias_austen_modelo_dict[token] = frecuencia

# Recontar Twain sobre la representacion final del entrenamiento.
tokens_twain_modelo = []
for oracion in twain_train_marcado:
    for token in oracion:
        tokens_twain_modelo.append(token)

tokens_twain_modelo, frecuencias_twain_modelo = np.unique(
    tokens_twain_modelo,
    return_counts=True
)

frecuencias_twain_modelo_dict = {}
for token, frecuencia in zip(tokens_twain_modelo, frecuencias_twain_modelo):
    frecuencias_twain_modelo_dict[token] = frecuencia

print("Primeras oraciones Austen con marcas:")
for oracion in austen_train_marcado[:2]:
    print(oracion)

print("\nFrecuencias finales de Austen:")
print("<UNK>:", frecuencias_austen_modelo_dict.get("<UNK>", 0))
print("<s>:", frecuencias_austen_modelo_dict.get("<s>", 0))
print("</s>:", frecuencias_austen_modelo_dict.get("</s>", 0))

Primeras oraciones Austen con marcas:
['<s>', 'transcriber', "'s", 'note', '<UNK>', '<UNK>', 'is', 'indicated', 'by', '<UNK>', '.', '</s>']
['<s>', '<UNK>', 'letters', 'are', 'indicated', 'thus', ':', 'm^r', ',', 'm^', '{', 'rs', '}', '.', '</s>']

Frecuencias finales de Austen:
<UNK>: 6579
<s>: 32478
</s>: 32478


In [70]:
resumen = pd.DataFrame([
    {
        "Dataset": "Train Jane Austen",
        "Libros": len(train_jane_tokens),
        "Oraciones": sum(len(libro) for libro in train_jane_tokens),
        "Tokens": sum(len(oracion) for libro in train_jane_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in train_jane_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in austen_train_unk for oracion in libro),
        "Vocabulario": len(vocabulario_austen)
    },
    {
        "Dataset": "Train Mark Twain",
        "Libros": len(train_twain_tokens),
        "Oraciones": sum(len(libro) for libro in train_twain_tokens),
        "Tokens": sum(len(oracion) for libro in train_twain_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in train_twain_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in twain_train_unk for oracion in libro),
        "Vocabulario": len(vocabulario_twain)
    },
    {
        "Dataset": "Test Jane Austen",
        "Libros": len(test_jane_tokens),
        "Oraciones": sum(len(libro) for libro in test_jane_tokens),
        "Tokens": sum(len(oracion) for libro in test_jane_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in test_jane_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in austen_test_unk for oracion in libro),
        "Vocabulario": "-"
    },
    {
        "Dataset": "Test Mark Twain",
        "Libros": len(test_twain_tokens),
        "Oraciones": sum(len(libro) for libro in test_twain_tokens),
        "Tokens": sum(len(oracion) for libro in test_twain_tokens for oracion in libro),
        "<NUM>": sum(oracion.count("<NUM>") for libro in test_twain_tokens for oracion in libro),
        "<UNK>": sum(oracion.count("<UNK>") for libro in twain_test_unk for oracion in libro),
        "Vocabulario": "-"
    }
])

display(resumen)

,Dataset,Libros,Oraciones,Tokens,<NUM>,<UNK>,Vocabulario
0,Train Jane Austen,9,32478,857807,1637,6579,10912
1,Train Mark Twain,44,140517,3305102,9113,22309,34867
2,Test Jane Austen,1,7493,192857,9,5333,-
3,Test Mark Twain,1,5961,136716,127,2200,-
